# Computer Vision — โจทย์แบบ "จำแนกรูปภาพ" (Image Classification)

รูป 1 รูป -> 1 ป้าย (เช่น บ้าน/ไม่ใช่บ้าน, สุนัข/แมว, ปกติ/ผิดปกติ, ชนิดสินค้า)

วิธีในโน้ตบุ๊กนี้ (ทำจากบนลงล่าง):
- วิธีที่ 1 (ง่ายสุด แนะนำมือใหม่ทำแค่นี้) = `AutoGluon` กดรันแล้วมันเลือกโมเดล+เทรนให้เอง
- วิธีที่ 2 (ไม่บังคับ ทำถ้าอยากได้คะแนนสูงขึ้น) = `timm` เลือกโมเดลเอง + TTA


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ: input เป็นรูป และต้องตอบว่า "รูปนี้คือคลาสอะไร" (ตอบเป็นป้าย/ตัวเลข ไม่ใช่ข้อความยาว)
ถ้าโจทย์ให้ "หากรอบวัตถุ (กล่อง)" -> ไปใช้ `object_detection.ipynb`
ถ้าโจทย์ให้ "บรรยายรูปเป็นข้อความ" -> ไปหัวข้อ 03 Multimodal

ต้องแก้อะไรบ้าง (หาคำว่า `# << แก้`): ชื่อ competition, path โฟลเดอร์รูป, ชื่อคอลัมน์, จำนวนคลาส

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install -U "autogluon.multimodal"        # วิธีที่ 1
!pip -q install timm torch torchvision pillow scikit-learn pandas numpy tqdm   # วิธีที่ 2

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "super-ai-engineer-season-6-individual-hackathon-house-recognition"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — ตั้งค่า path + CONFIG (ดูจาก output ด้านบนแล้วแก้ให้ตรง)

In [ ]:
import os, glob, pandas as pd, numpy as np

def find(name):   # หาไฟล์ตามชื่อ
    h = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True); return h[0] if h else None
def img_dir(keyword):   # หาโฟลเดอร์รูปที่ชื่อมี keyword (train/test)
    cand = [d for d,_,fs in os.walk(DATA_DIR)
            if keyword in d.lower() and any(f.lower().endswith((".jpg",".png",".jpeg")) for f in fs)]
    return max(cand, key=lambda d: len(os.listdir(d))) if cand else None

TRAIN_CSV  = find("train.csv")               # << แก้ถ้าหาไม่เจอ: ใส่ path เอง เช่น "data/train_labels.csv"
SAMPLE_SUB = find("sample_submission.csv")
TRAIN_IMG  = img_dir("train")                # << แก้ถ้าผิด: ใส่ path เอง เช่น "data/train_images"
TEST_IMG   = img_dir("test")                 # << แก้ถ้าผิด: เช่น "data/test_images"

df = pd.read_csv(TRAIN_CSV); sample = pd.read_csv(SAMPLE_SUB)
print("train.csv คอลัมน์:", list(df.columns)); print("sample คอลัมน์:", list(sample.columns))
print("รูป train:", TRAIN_IMG, "| รูป test:", TEST_IMG)
display(df.head()); display(sample.head())

IMG_COL    = "image_name"        # << แก้: ชื่อคอลัมน์ไฟล์รูปใน train.csv (ดูจาก "train.csv คอลัมน์:") เช่น "filename", "id"
LABEL_COL  = "class"             # << แก้: ชื่อคอลัมน์คำตอบใน train.csv เช่น "label", "target", "category"
ID_COL     = sample.columns[0]   # เดาอัตโนมัติ: คอลัมน์แรกของ sample คือ id
ANSWER_COL = sample.columns[1]   # เดาอัตโนมัติ: คอลัมน์ที่สองคือคำตอบ
TEST_EXT   = ".jpg"              # << แก้ถ้านามสกุลไฟล์ test ไม่ใช่ .jpg เช่น ".png" (ถ้าใน sample id มีนามสกุลอยู่แล้ว ใส่ "")
NUM_CLASSES = int(df[LABEL_COL].nunique())
print("จำนวนคลาส =", NUM_CLASSES)

## 🔎 โจทย์นี้ต้องส่งอะไร + วัดด้วยอะไร (รันก่อนทำ submission)

เซลล์นี้ตอบ 3 คำถามที่มือใหม่มักไม่รู้:
- ต้องเติม "คอลัมน์อะไร" ลง submission และเป็น "ชนิดไหน" (ดูจาก sample_submission)
- โจทย์วัดด้วย "metric อะไร" (ดึงจาก Kaggle ให้อัตโนมัติ)
- ต้องส่งเป็น "ป้าย / ความน่าจะเป็น / ตัวเลข" แบบไหน

In [ ]:
# บอกว่าต้องเติมอะไรลง submission + ตัวอย่างค่าที่ sample ให้มา
print("ไฟล์ส่งต้องมีคอลัมน์:", list(sample.columns), "| จำนวนแถว:", len(sample))
for _c in list(sample.columns)[1:]:
    print(f"  - เติม '{_c}': ชนิด {sample[_c].dtype}, ตัวอย่างค่าใน sample = {list(sample[_c].dropna().unique()[:3])}")
# ดึง metric จาก Kaggle อัตโนมัติ (ถ้าต่อ API ได้)
_metric = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    _api = KaggleApi(); _api.authenticate()
    _resp = _api.competitions_list(search=COMP)
    for _co in (getattr(_resp, "competitions", _resp) or []):   # kaggle ใหม่คืน response (ใช้ .competitions), เก่าคืน list
        if str(getattr(_co, "ref", "")).rstrip("/").split("/")[-1] == COMP:
            _metric = getattr(_co, "evaluation_metric", None) or getattr(_co, "evaluationMetric", None); break
except Exception:
    pass
def _how_to_send(m):
    m = (m or "").lower()
    if any(k in m for k in ["auc","roc","log loss","logloss","brier","probab"]): return "ความน่าจะเป็น (เลข 0-1)"
    if any(k in m for k in ["rmse","mae","mse","r2","rmsle","error","mape","smape"]): return "ตัวเลขต่อเนื่อง"
    return "ป้าย/คลาส (เช่น 0/1 หรือชื่อคลาส)"
print("\nMetric (ดึงจาก Kaggle):", _metric or "ดึงไม่ได้ -> เปิด tab Evaluation บนหน้า Kaggle อ่านเอง")
print("=> ปกติต้องส่งเป็น:", _how_to_send(_metric), "(เช็คกับ tab Evaluation อีกที)")

## วิธีที่ 1 — AutoGluon (ง่ายสุด มือใหม่ทำแค่นี้ก็ได้คะแนนดี)

แค่ทำตารางที่มี path รูป + ป้าย แล้วสั่ง fit เดี๋ยว AutoGluon เลือกโมเดล + เทรน + รวมโมเดลให้เอง
ปรับ `time_limit` (วินาที) ตามเวลาที่มี

In [ ]:
from autogluon.multimodal import MultiModalPredictor

train_df = df.copy()
train_df["image"] = train_df[IMG_COL].apply(lambda n: os.path.join(TRAIN_IMG, str(n)))
predictor = MultiModalPredictor(label=LABEL_COL, eval_metric="accuracy", path="ag_imgcls")
predictor.fit(train_df[["image", LABEL_COL]], time_limit=900)   # << แก้ time_limit: วินาที (900=15นาที) ลอง 300 ก่อน, ส่งจริงเพิ่มเป็น 1800-3600

test_df = sample.copy()
test_df["image"] = test_df[ID_COL].apply(lambda i: os.path.join(TEST_IMG, str(i)+TEST_EXT))
out = sample.copy(); out[ANSWER_COL] = predictor.predict(test_df[["image"]]).values   # ไม่ astype(int) เพื่อรองรับป้ายที่เป็นข้อความด้วย
out.to_csv("submission.csv", index=False)
print("บันทึก submission.csv เรียบร้อย"); display(out.head())

## วิธีที่ 2 — timm fine-tune + TTA (ไม่บังคับ ทำถ้าอยากได้คะแนนสูงขึ้น ต้องมี GPU)

โหลดโมเดล pretrained จาก `timm` มาเทรนต่อ + augmentation + TTA
เลือกโมเดลอื่นได้ที่ `MODEL_NAME` (ดูตัวเลือกใน reference_cheatsheet.md)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from PIL import Image
from tqdm.auto import tqdm

MODEL_NAME="tf_efficientnetv2_s.in21k_ft_in1k"  # << แก้โมเดลได้ เช่น "resnet50" (เบา/เร็ว), "convnext_small.fb_in22k_ft_in1k" (แม่น)
IMG_SIZE=300; EPOCHS=12; BATCH=32; LR=3e-4      # << แก้ตามเวลา/GPU
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; AMP=(DEVICE=="cuda")
MEAN,STD=[0.485,0.456,0.406],[0.229,0.224,0.225]
tf_tr=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(.3,.3,.3,.1),transforms.RandomRotation(15),transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),transforms.RandomErasing(p=0.2)])
tf_ev=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
tf_tta=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
class DS(Dataset):
    def __init__(s,frame,d,tf,test=False,idc=None): s.f=frame.reset_index(drop=True);s.d=d;s.tf=tf;s.t=test;s.idc=idc
    def __len__(s): return len(s.f)
    def __getitem__(s,i):
        r=s.f.iloc[i]
        if s.t: return s.tf(Image.open(os.path.join(s.d,str(r[s.idc])+TEST_EXT)).convert("RGB")),0
        return s.tf(Image.open(os.path.join(s.d,str(r[IMG_COL]))).convert("RGB")),int(r[LABEL_COL])
skf=StratifiedKFold(5,shuffle=True,random_state=42)
tr,va=next(iter(skf.split(df,df[LABEL_COL])))
model=timm.create_model(MODEL_NAME,pretrained=True,num_classes=NUM_CLASSES,drop_rate=0.3).to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
crit=nn.CrossEntropyLoss(label_smoothing=0.1); scaler=torch.amp.GradScaler("cuda", enabled=AMP)
dl=DataLoader(DS(df.iloc[tr],TRAIN_IMG,tf_tr),BATCH,shuffle=True,num_workers=2,drop_last=True)
vl=DataLoader(DS(df.iloc[va],TRAIN_IMG,tf_ev),BATCH,num_workers=2)
for ep in range(EPOCHS):
    model.train()
    for x,y in tqdm(dl,desc=f"ep{ep+1}/{EPOCHS}",leave=False):
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad()
        with torch.amp.autocast(device_type=DEVICE, enabled=AMP): loss=crit(model(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sch.step()
    model.eval(); cc=tt=0
    with torch.no_grad():
        for x,y in vl:
            with torch.amp.autocast(device_type=DEVICE, enabled=AMP): pr=model(x.to(DEVICE)).argmax(1).cpu()
            cc+=(pr==y).sum().item(); tt+=len(y)
    print(f"ep{ep+1} val_acc={cc/tt:.4f}")
probs=np.zeros((len(sample),NUM_CLASSES))
for s_ in range(5):
    tf=tf_ev if s_==0 else tf_tta
    dl=DataLoader(DS(sample,TEST_IMG,tf,test=True,idc=ID_COL),BATCH,num_workers=2)
    o=[]
    with torch.no_grad():
        for x,_ in tqdm(dl,desc=f"TTA{s_+1}",leave=False):
            with torch.amp.autocast(device_type=DEVICE, enabled=AMP): o.append(F.softmax(model(x.to(DEVICE)),1).float().cpu().numpy())
    probs+=np.vstack(o)
out=sample.copy(); out[ANSWER_COL]=probs.argmax(1).astype(int)
out.to_csv("submission_timm.csv",index=False); print("บันทึก submission_timm.csv"); display(out.head())

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "image classification"
!kaggle competitions submissions -c {COMP} | head